In [16]:
# plot_mais_curves.py
import numpy as np
import matplotlib.pyplot as plt
import csv
from pathlib import Path
import matplotlib.ticker as mticker

# ----------------------------- Config -----------------------------
# X-axis range (mph)
DV_MIN, DV_MAX, DV_STEP = 0, 65, 0.25
EXPORT_CSV = True  # set False if you don't want CSVs

# Work in the same folder as this script
try:
    OUTDIR = Path(__file__).parent.resolve()
except NameError:
    # Fallback for Jupyter/interactive use
    OUTDIR = Path.cwd().resolve()

# Base font boost for readability in papers
plt.rcParams.update({
    "font.size": 14,
    "figure.dpi": 150,
    "savefig.dpi": 300
})

# Colour-blind-friendly palette (matplotlib tab10)
COLS = {
    "MAIS 1+": "#1f77b4",   # blue (dashed)
    "MAIS 2+": "#d62728",   # red
    "MAIS 3+": "#2ca02c",   # green (dashed)
    "MAIS 4+": "#9467bd",   # purple
    "MAIS 5+": "#17becf",   # teal
    "Fatality": "#ff7f0e",  # orange
}

LINESTYLES = {
    "MAIS 1+": "--",
    "MAIS 3+": "--",
    # others solid by default
}

LINEWIDTH = 2.5

# -------------------------- Probability model ---------------------
def logistic(a, b, D):
    """Return logistic probability: exp(a + bD) / (1 + exp(a + bD))."""
    x = a + b * D
    ex = np.exp(x)
    return ex / (1.0 + ex)

COEFFS_FRONTAL = {
    "MAIS 1+": (-1.4930, 0.0854),
    "MAIS 2+": (-4.9429, 0.1425),
    "MAIS 3+": (-6.9774, 0.1620),
    "MAIS 4+": (-8.4254, 0.1586),
    "MAIS 5+": (-8.8355, 0.1566),
    "Fatality": (-9.0422, 0.1571),
}

COEFFS_REAR = {
    "MAIS 1+": (-1.8199, 0.0671),
    "MAIS 2+": (-6.1818, 0.1482),
    "MAIS 3+": (-8.0329, 0.1793),
    "MAIS 4+": (-11.8787, 0.2210),
    "MAIS 5+": (-12.1944, 0.2276),
    "Fatality": (-12.1982, 0.2255),
}

CURVE_ORDER = ["MAIS 1+", "MAIS 2+", "MAIS 3+", "MAIS 4+", "MAIS 5+", "Fatality"]

# ---------------------------- Plot helper -------------------------
def add_kmh_top_axis(ax):
    """Add a secondary top x-axis in km/h."""
    mph_to_kmh = 1.60934
    def mph2kmh(x): return x * mph_to_kmh
    def kmh2mph(x): return x / mph_to_kmh
    secax = ax.secondary_xaxis('top', functions=(mph2kmh, kmh2mph))
    secax.set_xlabel("Delta-v (km/h)")
    return secax

def plot_panel(ax, D_mph, coeffs, title):
    for label in CURVE_ORDER:
        a, b = coeffs[label]
        y = logistic(a, b, D_mph)
        ls = LINESTYLES.get(label, "-")
        ax.plot(D_mph, y, ls=ls, lw=LINEWIDTH, color=COLS[label], label=label)

    ax.set_xlim(DV_MIN, DV_MAX)
    ax.set_ylim(0.0, 1.0)   # stretch a bit for breathing space
    ax.set_xlabel("Delta-v (mph)")
    ax.set_ylabel("Probability")

    # Force grid spacing
    # ax.xaxis.set_major_locator(mticker.MultipleLocator(5))
    ax.yaxis.set_major_locator(mticker.MultipleLocator(0.1))

    ax.grid(True, which="both", linestyle=":", linewidth=0.8, alpha=0.7)

    add_kmh_top_axis(ax)

    # Add the "title" as text inside plot area (top-left)
    ax.text(
        0.03, 0.97, title, 
        transform=ax.transAxes,
        ha="left", va="top",
        fontsize=14, 
        bbox=dict(facecolor="white", edgecolor="black", boxstyle="round,pad=0.3", alpha=0.8)
    )

def export_csv(D_mph, coeffs, outpath_csv):
    header = ["Delta_v_mph", "Delta_v_kmh"] + CURVE_ORDER
    mph_to_kmh = 1.60934
    with open(outpath_csv, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(header)
        for D in D_mph:
            row = [f"{D:.2f}", f"{D*mph_to_kmh:.2f}"]
            for label in CURVE_ORDER:
                a, b = coeffs[label]
                row.append(f"{logistic(a, b, D):.8f}")
            writer.writerow(row)

# ------------------------------ Main ------------------------------
if __name__ == "__main__":
    D = np.arange(DV_MIN, DV_MAX + DV_STEP, DV_STEP)

    # Make the figure taller (more vertical space)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 7.5), constrained_layout=True)

    plot_panel(ax1, D, COEFFS_FRONTAL, "Frontal")
    plot_panel(ax2, D, COEFFS_REAR,    "Rear-end")

    # Legend moved further down to avoid clashing with x-axis label
    handles, labels = ax1.get_legend_handles_labels()
    fig.legend(
        handles, labels,
        loc="lower center",
        ncol=6,
        frameon=False,
        bbox_to_anchor=(0.5, -0.05),   # <-- push further down
        fontsize=14
)

    pdf_path = OUTDIR / "mais_curves.pdf"
    png_path = OUTDIR / "mais_curves.png"
    fig.savefig(pdf_path, bbox_inches="tight")
    fig.savefig(png_path, bbox_inches="tight")
    plt.close(fig)

    if EXPORT_CSV:
        export_csv(D, COEFFS_FRONTAL, OUTDIR / "mais_curves_frontal.csv")
        export_csv(D, COEFFS_REAR,    OUTDIR / "mais_curves_rear.csv")

    print(f"Saved in script folder: {pdf_path}, {png_path}")
    if EXPORT_CSV:
        print("Saved CSVs:", OUTDIR / "mais_curves_frontal.csv", OUTDIR / "mais_curves_rear.csv")


Saved in script folder: D:\work\sumo-paper-grid-clean\sumo-safety-traci-project\analysis\mais_curves.pdf, D:\work\sumo-paper-grid-clean\sumo-safety-traci-project\analysis\mais_curves.png
Saved CSVs: D:\work\sumo-paper-grid-clean\sumo-safety-traci-project\analysis\mais_curves_frontal.csv D:\work\sumo-paper-grid-clean\sumo-safety-traci-project\analysis\mais_curves_rear.csv
